In [6]:

import requests

base = "https://datatalks.club/faq"
courses = requests.get(f"{base}/json/courses.json").json()

documents = []
for course in courses:
    course_data = requests.get(f"{base}/{course['path']}").json()
    documents.extend(course_data)

len(documents)

1302

In [7]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["question", "answer", "section"],
    keyword_fields=["course"],
)

index.fit(documents)


In [8]:
def search(query, limit=5):
    """Search the FAQ for one course using the in-memory minsearch index.

    Args:
        query: Student question to look up.
        limit: Maximum number of matching FAQ entries to return.

    Returns:
        A list of matching FAQ documents.
    """
    course = 'data-engineering-zoomcamp'

    return index.search(
        query=query,
        filter_dict={"course": course},
        boost_dict={"question": 3.0, "section": 0.5, "answer": 1.0},
        num_results=limit,
    )


In [9]:
import json
from openai import AsyncOpenAI

openai_client = AsyncOpenAI()
MODEL_NAME = "gpt-4o-mini"


In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course FAQ.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string"},
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}


In [11]:
INSTRUCTIONS = """
You're a teaching assistant for DataTalks.Club zoomcamps.

Answer the user's question using the FAQ knowledge base. Use the `search`
tool to look things up. You can call search multiple times with different
queries to explore the topic well.

Rules:
- Use only facts from the search results.
- If the answer isn't in the results, say so clearly.
- At the end, list the FAQ entries you used under a "Sources" section,
  one per line in the form: `- [id] section > question`.
""".strip()

In [12]:
question = "I just discovered the course, can I still join?"

message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": question},
]


In [13]:
message_history

[{'role': 'system',
  'content': 'You\'re a teaching assistant for DataTalks.Club zoomcamps.\n\nAnswer the user\'s question using the FAQ knowledge base. Use the `search`\ntool to look things up. You can call search multiple times with different\nqueries to explore the topic well.\n\nRules:\n- Use only facts from the search results.\n- If the answer isn\'t in the results, say so clearly.\n- At the end, list the FAQ entries you used under a "Sources" section,\n  one per line in the form: `- [id] section > question`.'},
 {'role': 'user',
  'content': 'I just discovered the course, can I still join?'}]

In [14]:
async with openai_client.responses.stream(
    model=MODEL_NAME,
    input=message_history,
    tools=[search_tool],
) as stream:
    response = await stream.get_final_response()


In [15]:
for item in response.output:
    if item.type != "function_call":
        continue

    args = json.loads(item.arguments)
    result = search(**args)

    message_history.append({
        "type": "function_call",
        "call_id": item.call_id,
        "name": item.name,
        "arguments": item.arguments,
    })

    message_history.append({
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": json.dumps(result),
    })


In [16]:
message_history

[{'role': 'system',
  'content': 'You\'re a teaching assistant for DataTalks.Club zoomcamps.\n\nAnswer the user\'s question using the FAQ knowledge base. Use the `search`\ntool to look things up. You can call search multiple times with different\nqueries to explore the topic well.\n\nRules:\n- Use only facts from the search results.\n- If the answer isn\'t in the results, say so clearly.\n- At the end, list the FAQ entries you used under a "Sources" section,\n  one per line in the form: `- [id] section > question`.'},
 {'role': 'user',
  'content': 'I just discovered the course, can I still join?'},
 {'type': 'function_call',
  'call_id': 'call_CYr3icidXRflevahR4lwGnOw',
  'name': 'search',
  'arguments': '{"query":"can I still join the course"}'},
 {'type': 'function_call_output',
  'call_id': 'call_CYr3icidXRflevahR4lwGnOw',
  'output': '[{"id": "3f1424af17", "course": "data-engineering-zoomcamp", "section": "General Course-Related Questions", "question": "Course: Can I still join th

In [17]:
async with openai_client.responses.stream(
    model=MODEL_NAME,
    input=message_history,
    tools=[search_tool],
) as stream:
    async for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

    response = await stream.get_final_response()


Yes, you can still join the course even if it's already started. You're eligible to submit homework, but be aware that there will be deadlines for submitting homework and final projects, so it's important not to leave everything until the last minute.

If you're looking for a certificate, note that you can only receive one if you finish the course with a "live" cohort. Self-paced learners do not receive certificates.

Sources:
- [3f1424af17] General Course-Related Questions > Course: Can I still join the course after the start date?
- [900f60fd25] General Course-Related Questions > Certificate - Can I follow the course in a self-paced mode and get a certificate?

In [18]:
class NotebookRenderer:
    async def handle_event(self, event_type, payload):
        handler = getattr(self, f"handle_{event_type}", self.handle_unknown)
        await handler(payload)

    async def handle_status(self, payload):
        print(f"[{payload['message']}]")

    async def handle_iteration(self, payload):
        print(f"\n--- iteration {payload['n']} ---")

    async def handle_tool_call(self, payload):
        print(f"\n[tool_call] {payload['name']}({payload['arguments']})")

    async def handle_tool_result(self, payload):
        count = len(payload["result"]) if isinstance(payload["result"], list) else "?"
        print(f"[tool_result] {payload['name']} -> {count} hits")

    async def handle_token(self, payload):
        print(payload["delta"], end="", flush=True)

    async def handle_done(self, payload):
        print(f"\n\n[done]\n{payload['answer']}")

    async def handle_unknown(self, payload):
        print(payload)


renderer = NotebookRenderer()

In [19]:
async def request_response(message_history, renderer):
    async with openai_client.responses.stream(
        model=MODEL_NAME,
        input=message_history,
        tools=[search_tool],
    ) as stream:
        async for event in stream:
            if event.type == "response.output_text.delta":
                await renderer.handle_event("token", {"delta": event.delta})

        return await stream.get_final_response()


In [20]:
def append_tool_messages(message_history, item, result):
    message_history.append(
        {
            "type": "function_call",
            "call_id": item.call_id,
            "name": item.name,
            "arguments": item.arguments,
        }
    )

    message_history.append(
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result),
        }
    )


In [21]:
async def handle_tool_calls(response, message_history, renderer):
    has_tool_calls = False

    for item in response.output:
        if item.type != "function_call":
            continue

        has_tool_calls = True

        args = json.loads(item.arguments)
        await renderer.handle_event(
            "tool_call",
            {"name": item.name, "arguments": args},
        )

        result = search(**args)
        preview = [
            {
                "id": hit.get("id"),
                "course": hit.get("course"),
                "question": hit.get("question"),
            }
            for hit in result
        ]
        await renderer.handle_event(
            "tool_result",
            {"name": item.name, "result": preview},
        )

        append_tool_messages(message_history, item, result)

    return has_tool_calls


def collect_answer(response):
    answer = ""

    for item in response.output:
        if item.type != "message":
            continue

        for content in item.content:
            if getattr(content, "text", None):
                answer += content.text

    return answer


In [22]:
MAX_ITERATIONS = 5


async def run_agent(question, renderer):
    await renderer.handle_event("status", {"message": "thinking..."})
    message_history = [
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": question},
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        await renderer.handle_event("iteration", {"n": iteration})

        response = await request_response(message_history, renderer)
        has_tool_calls = await handle_tool_calls(response, message_history, renderer)

        if not has_tool_calls:
            answer = collect_answer(response)
            await renderer.handle_event("done", {"answer": answer})
            return

    await renderer.handle_event("done", {"answer": "(stopped: reached max iterations)"})


In [23]:
await run_agent("I just discovered the course, can I still join?", renderer)


[thinking...]

--- iteration 1 ---

[tool_call] search({'query': 'Can I still join the course'})
[tool_result] search -> 5 hits

--- iteration 2 ---
Yes, you can still join the course even after its start date. Although you won't be registered, you can submit homework assignments. Keep in mind that there are deadlines for submitting homework and final projects, so it’s best not to leave everything until the last minute.

If you’re looking to join an upcoming cohort, the next one starts on January 12th, 2026. Make sure to register before it starts.

### Sources
- [3f1424af17] General Course-Related Questions > Can I still join the course after the start date?
- [9e508f2212] General Course-Related Questions > When does the course start?

[done]
Yes, you can still join the course even after its start date. Although you won't be registered, you can submit homework assignments. Keep in mind that there are deadlines for submitting homework and final projects, so it’s best not to leave everyt